# Learned Stage-Wise Curvature Tuning on Arabic Characters

This notebook evaluates Learned Stage-Wise Curvature Tuning (SW-CT) on the Arabic Handwritten Characters dataset using an ImageNet-pretrained ResNet-18.

Arabic Characters is a handwritten character-recognition task that differs substantially from the natural-image domain used for ImageNet pretraining. This provides a strong domain-shift experiment for evaluating whether stage-wise curvature adaptation can transfer to a visually different classification problem.

The experiments compare four conditions:

1. **Baseline** — the original pretrained ResNet with ReLU activations and a trained downstream classifier.
2. **Single-Parameter Curvature Tuning (S-CT)** — one global beta selected through validation-based exhaustive search.
3. **Learned SW-CT with fixed initialization** — four trainable stage-wise beta parameters initialized at 0.78.
4. **Learned SW-CT with random initialization** — four trainable stage-wise beta parameters independently initialized between 0.70 and 0.99.

Each condition is evaluated using seeds 42, 43, and 44.

Experiments are conducted for both 20 and 30 epochs to determine whether additional optimization improves classification performance or changes the learned stage-wise beta values.

Wall-clock execution time is also recorded to compare computational cost across Baseline, S-CT, and Learned SW-CT.

In [ ]:
!nvidia-smi

In [ ]:
%cd /content

!git clone -b stage-wise-ct \
    https://github.com/Syed1611/curvature-tuning-research.git

%cd /content/curvature-tuning-research/src/curvature-tuning

!git branch --show-current
!git log -1 --oneline

## Environment Setup

The repository dependencies are installed before running the experiments.

After installation, the Colab runtime should be restarted so that the required package versions are loaded correctly.

In [ ]:
%pip install -q -r requirements.txt
%pip install --force-reinstall --no-cache-dir "numpy==1.26.3"

In [1]:
%cd /content/curvature-tuning-research/src/curvature-tuning

import os
import torch

os.environ["WANDB_MODE"] = "disabled"

print("Working directory:", os.getcwd())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/content/curvature-tuning-research/src/curvature-tuning
Working directory: /content/curvature-tuning-research/src/curvature-tuning
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## Arabic Characters Dataset

Before running the experiments, the Arabic Characters data loaders are checked to confirm that the downstream dataset can be loaded correctly.

The same dataset split and random seed protocol are used for Baseline, S-CT, and Learned SW-CT so that the methods are compared under the same conditions.

In [2]:
from utils.data import get_data_loaders

train_loader, test_loader, val_loader = get_data_loaders(
    "imagenet_to_arabic-characters",
    train_batch_size=32,
    test_batch_size=128,
    seed=42,
)

print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels[:10])

README.md: 0.00B [00:00, ?B/s]

arabic-characters.py: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Processing train set: 100%|██████████| 13441/13441 [00:04<00:00, 3004.56it/s]


Generating test split: 0 examples [00:00, ? examples/s]

Processing test set: 100%|██████████| 3361/3361 [00:01<00:00, 2431.79it/s]


Train samples: 10752
Validation samples: 2688
Test samples: 3360
Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Labels: tensor([ 9, 16, 17, 14,  6, 24, 15, 10, 26, 15])


# ResNet-18 Experiments

ResNet-18 is evaluated first because it was used during the initial development of Learned SW-CT.

The experiment compares Baseline, S-CT, fixed-initialization SW-CT, and random-initialization SW-CT.

All methods use the same ImageNet-pretrained backbone, Arabic Characters dataset split, experimental seeds, and downstream training protocol.

## ResNet-18 — 20-Epoch Evaluation

The first evaluation uses 20 training epochs.

This provides the primary reference experiment and maintains compatibility with the earlier Learned SW-CT work.

### Methods

**Baseline**

The pretrained ResNet-18 retains its original ReLU activations. The backbone is frozen and only the downstream classifier is trained.

**S-CT**

The original Single-Parameter Curvature Tuning method searches over a single beta shared throughout the network. Candidate beta values from 0.70 to 1.00 are evaluated using validation performance, and the best candidate is evaluated on the test set.

The recorded S-CT execution time includes the complete beta search rather than only the final selected beta.

**Learned SW-CT — Fixed Initialization**

Four trainable beta parameters are assigned to the four ResNet stages. All four begin at beta = 0.78.

This preserves the initialization used in the original Learned SW-CT experiments.

**Learned SW-CT — Random Initialization**

Four trainable stage-wise beta parameters are initialized independently between 0.70 and 0.99.

The random initialization is generated deterministically from the experiment seed. This allows the experiment to test whether Learned SW-CT depends on the fixed 0.78 starting value while remaining reproducible.

In [3]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-02 00:02:36.892 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_arabic-characters_resnet18_seed42_epochs20.log
2026-10-02 00:02:36.895 | INFO     | __main__:main:78 - Running on cuda
2026-10-02 00:03:00.301 | INFO     | __main__:main:105 - Testing baseline...
2026-10-02 00:03:00.315 | INFO     | __main__:main:116 - Number of trainable parameters: 14364
2026-10-02 00:03:00.315 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-02 00:03:18.153 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.350898, Accuracy: 6.25%
2026-10-02 00:03:18.461 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 3.325150, Accuracy: 6.37%
2026-10-02 00:03:18.785 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 3.128802, Accuracy: 16.18%
2026-10-02 00:03:19.103 | INFO     | train:train_epoch:47 - Epoch 1, Step 300, Loss: 2.893454, Accuracy: 25.38%
2026-10-02 00:03:19.387 | INFO     | train:test_epoch:90 - Epoch 1, Val Los

In [4]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-02 00:33:12.019 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_arabic-characters_resnet18_seed43_epochs20.log
2026-10-02 00:33:12.021 | INFO     | __main__:main:78 - Running on cuda
2026-10-02 00:33:34.429 | INFO     | __main__:main:105 - Testing baseline...
2026-10-02 00:33:34.449 | INFO     | __main__:main:116 - Number of trainable parameters: 14364
2026-10-02 00:33:34.449 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-02 00:33:55.048 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.390915, Accuracy: 6.25%
2026-10-02 00:33:55.377 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 3.315051, Accuracy: 7.12%
2026-10-02 00:33:55.721 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 3.124616, Accuracy: 16.23%
2026-10-02 00:33:56.050 | INFO     | train:train_epoch:47 - Epoch 1, Step 300, Loss: 2.887744, Accuracy: 26.24%
2026-10-02 00:33:56.389 | INFO     | train:test_epoch:90 - Epoch 1, Val Los

In [5]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-02 01:04:36.622 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_arabic-characters_resnet18_seed44_epochs20.log
2026-10-02 01:04:36.623 | INFO     | __main__:main:78 - Running on cuda
2026-10-02 01:04:56.401 | INFO     | __main__:main:105 - Testing baseline...
2026-10-02 01:04:56.417 | INFO     | __main__:main:116 - Number of trainable parameters: 14364
2026-10-02 01:04:56.417 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-02 01:05:16.791 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.360070, Accuracy: 6.25%
2026-10-02 01:05:17.101 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 3.323182, Accuracy: 8.57%
2026-10-02 01:05:17.410 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 3.132338, Accuracy: 17.52%
2026-10-02 01:05:17.715 | INFO     | train:train_epoch:47 - Epoch 1, Step 300, Loss: 2.897824, Accuracy: 26.25%
2026-10-02 01:05:18.035 | INFO     | train:test_epoch:90 - Epoch 1, Val Los

### Learned SW-CT with Fixed Beta Initialization

The original Learned SW-CT configuration is first reproduced using the fixed initialization:

\[[\beta_1,\beta_2,\beta_3,\beta_4]=[0.78,0.78,0.78,0.78].\]

The initial value is only a starting point. During training, the four beta parameters are optimized independently together with the downstream classifier.

These runs provide the reference against which random initialization will be compared.

In [6]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-02 01:34:12.167 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_arabic-characters_resnet18_seed42_epochs20_initfixed0.78_betalr0.01.log
2026-10-02 01:34:12.241 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 01:34:32.453 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-02 01:34:32.454 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-02 01:34:32.480 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-02 01:34:32.481 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 01:34:32.481 | INFO     | __main__:main:439 - Total trainable parameters: 14368
2026-10-02 01:34:33.387 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.489472, Accuracy: 6.25%
2026-10-02 01:34:48.921 | INFO     | train:train_e

In [7]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-02 01:54:14.934 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_arabic-characters_resnet18_seed43_epochs20_initfixed0.78_betalr0.01.log
2026-10-02 01:54:14.991 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 01:54:34.710 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-02 01:54:34.710 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-02 01:54:34.739 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-02 01:54:34.740 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 01:54:34.740 | INFO     | __main__:main:439 - Total trainable parameters: 14368
2026-10-02 01:54:35.639 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.476303, Accuracy: 3.12%
2026-10-02 01:54:51.137 | INFO     | train:train_e

In [8]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-02 02:14:17.156 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_arabic-characters_resnet18_seed44_epochs20_initfixed0.78_betalr0.01.log
2026-10-02 02:14:17.215 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 02:14:36.925 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-02 02:14:36.925 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-02 02:14:36.952 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-02 02:14:36.953 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 02:14:36.953 | INFO     | __main__:main:439 - Total trainable parameters: 14368
2026-10-02 02:14:37.824 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.459690, Accuracy: 6.25%
2026-10-02 02:14:53.386 | INFO     | train:train_e

### Learned SW-CT with Random Beta Initialization

The fixed beta = 0.78 initialization may raise the question of whether Learned SW-CT performs well only because its starting point was informed by the earlier S-CT experiments.

To test this dependency, each of the four stage-wise beta parameters is now initialized independently from a uniform distribution between 0.70 and 0.99.

The initialization is reproducible: the random generator is seeded using the experiment seed. Therefore, each seed always receives the same initial beta vector.

If random initialization produces similar performance to the fixed 0.78 experiment, this provides evidence that the Learned SW-CT result is not dependent on a single hand-selected initialization.

In [9]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-02 02:34:28.166 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_arabic-characters_resnet18_seed42_epochs20_initrandom_betalr0.01.log
2026-10-02 02:34:28.224 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 02:34:47.903 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-02 02:34:47.903 | INFO     | __main__:main:394 - Requested initial stage betas: [0.924447, 0.827275, 0.948993, 0.902237]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-02 02:34:47.939 | INFO     | __main__:main:418 - Initial stage betas: [0.924447238445282, 0.827274739742279, 0.9489933848381042, 0.9022367000579834]
2026-10-02 02:34:47.940 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 02:34:47.941 | INFO     | __main__:main:439 - Total trainable parameters: 14368
2026-10-02 02:34:49.001 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.494678, Accuracy: 9.38%
2026-10-02 02:35:04.699 | INFO     | t

In [10]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-02 02:54:40.933 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_arabic-characters_resnet18_seed43_epochs20_initrandom_betalr0.01.log
2026-10-02 02:54:41.002 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 02:55:04.286 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-02 02:55:04.286 | INFO     | __main__:main:394 - Requested initial stage betas: [0.889167, 0.712695, 0.705809, 0.943372]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-02 02:55:04.326 | INFO     | __main__:main:418 - Initial stage betas: [0.8891667127609253, 0.7126947641372681, 0.7058085799217224, 0.9433716535568237]
2026-10-02 02:55:04.327 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 02:55:04.327 | INFO     | __main__:main:439 - Total trainable parameters: 14368
2026-10-02 02:55:05.255 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.464369, Accuracy: 0.00%
2026-10-02 02:55:20.805 | INFO     |

In [11]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-02 03:14:55.610 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_arabic-characters_resnet18_seed44_epochs20_initrandom_betalr0.01.log
2026-10-02 03:14:55.667 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 03:15:18.546 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-02 03:15:18.547 | INFO     | __main__:main:394 - Requested initial stage betas: [0.735544, 0.774853, 0.817674, 0.981063]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-02 03:15:18.572 | INFO     | __main__:main:418 - Initial stage betas: [0.7355440258979797, 0.7748528122901917, 0.8176735043525696, 0.9810633659362793]
2026-10-02 03:15:18.572 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 03:15:18.573 | INFO     | __main__:main:439 - Total trainable parameters: 14368
2026-10-02 03:15:19.453 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.488366, Accuracy: 3.12%
2026-10-02 03:15:34.983 | INFO     |

# ResNet-18 — 30-Epoch Evaluation

The same experiments are now repeated for 30 epochs.

The purpose of increasing the training budget is to determine whether the 20-epoch results are limited by insufficient optimization.

If a method improves substantially with 30 epochs, this suggests that additional optimization is useful. If performance remains similar, it indicates that the method has largely stabilized within the original 20-epoch training budget.

All other experimental settings remain unchanged.

For random-initialization SW-CT, the same seed produces exactly the same initial four-beta vector in both the 20-epoch and 30-epoch experiments. Therefore, initialization is held constant while the training duration changes.

Execution time is again recorded to measure the computational cost of extending training from 20 to 30 epochs.

In [12]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-02 03:35:06.329 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_arabic-characters_resnet18_seed42_epochs30.log
2026-10-02 03:35:06.331 | INFO     | __main__:main:78 - Running on cuda
2026-10-02 03:35:25.841 | INFO     | __main__:main:105 - Testing baseline...
2026-10-02 03:35:25.855 | INFO     | __main__:main:116 - Number of trainable parameters: 14364
2026-10-02 03:35:25.855 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-02 03:35:46.649 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.350898, Accuracy: 6.25%
2026-10-02 03:35:46.958 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 3.325150, Accuracy: 6.37%
2026-10-02 03:35:47.260 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 3.128802, Accuracy: 16.18%
2026-10-02 03:35:47.566 | INFO     | train:train_epoch:47 - Epoch 1, Step 300, Loss: 2.893454, Accuracy: 25.38%
2026-10-02 03:35:47.878 | INFO     | train:test_epoch:90 - Epoch 1, Val Los

In [13]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-02 04:12:12.155 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_arabic-characters_resnet18_seed43_epochs30.log
2026-10-02 04:12:12.157 | INFO     | __main__:main:78 - Running on cuda
2026-10-02 04:12:32.857 | INFO     | __main__:main:105 - Testing baseline...
2026-10-02 04:12:32.873 | INFO     | __main__:main:116 - Number of trainable parameters: 14364
2026-10-02 04:12:32.873 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-02 04:12:53.251 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.390915, Accuracy: 6.25%
2026-10-02 04:12:53.555 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 3.315051, Accuracy: 7.12%
2026-10-02 04:12:53.865 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 3.124616, Accuracy: 16.23%
2026-10-02 04:12:54.188 | INFO     | train:train_epoch:47 - Epoch 1, Step 300, Loss: 2.887744, Accuracy: 26.24%
2026-10-02 04:12:54.508 | INFO     | train:test_epoch:90 - Epoch 1, Val Los

In [14]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-02 04:49:20.644 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_arabic-characters_resnet18_seed44_epochs30.log
2026-10-02 04:49:20.646 | INFO     | __main__:main:78 - Running on cuda
2026-10-02 04:49:42.156 | INFO     | __main__:main:105 - Testing baseline...
2026-10-02 04:49:42.169 | INFO     | __main__:main:116 - Number of trainable parameters: 14364
2026-10-02 04:49:42.170 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-02 04:50:02.323 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.360070, Accuracy: 6.25%
2026-10-02 04:50:02.600 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 3.323182, Accuracy: 8.57%
2026-10-02 04:50:02.911 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 3.132338, Accuracy: 17.52%
2026-10-02 04:50:03.220 | INFO     | train:train_epoch:47 - Epoch 1, Step 300, Loss: 2.897824, Accuracy: 26.25%
2026-10-02 04:50:03.602 | INFO     | train:test_epoch:90 - Epoch 1, Val Los

In [15]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-02 05:26:03.739 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_arabic-characters_resnet18_seed42_epochs30_initfixed0.78_betalr0.01.log
2026-10-02 05:26:03.790 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 05:26:19.653 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-02 05:26:19.653 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-02 05:26:19.679 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-02 05:26:19.679 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 05:26:19.679 | INFO     | __main__:main:439 - Total trainable parameters: 14368
2026-10-02 05:26:20.422 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.489472, Accuracy: 6.25%
2026-10-02 05:26:35.896 | INFO     | train:train_e

In [16]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-02 05:55:22.824 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_arabic-characters_resnet18_seed43_epochs30_initfixed0.78_betalr0.01.log
2026-10-02 05:55:22.883 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 05:55:42.388 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-02 05:55:42.388 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-02 05:55:42.415 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-02 05:55:42.415 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 05:55:42.416 | INFO     | __main__:main:439 - Total trainable parameters: 14368
2026-10-02 05:55:43.209 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.476303, Accuracy: 3.12%
2026-10-02 05:55:58.523 | INFO     | train:train_e

In [17]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-02 06:24:33.122 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_arabic-characters_resnet18_seed44_epochs30_initfixed0.78_betalr0.01.log
2026-10-02 06:24:33.173 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 06:24:49.063 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-02 06:24:49.063 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-02 06:24:49.088 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-02 06:24:49.088 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 06:24:49.089 | INFO     | __main__:main:439 - Total trainable parameters: 14368
2026-10-02 06:24:49.877 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.459690, Accuracy: 6.25%
2026-10-02 06:25:05.267 | INFO     | train:train_e

In [18]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-02 06:53:39.808 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_arabic-characters_resnet18_seed42_epochs30_initrandom_betalr0.01.log
2026-10-02 06:53:39.860 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 06:53:55.876 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-02 06:53:55.877 | INFO     | __main__:main:394 - Requested initial stage betas: [0.924447, 0.827275, 0.948993, 0.902237]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-02 06:53:55.904 | INFO     | __main__:main:418 - Initial stage betas: [0.924447238445282, 0.827274739742279, 0.9489933848381042, 0.9022367000579834]
2026-10-02 06:53:55.905 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 06:53:55.905 | INFO     | __main__:main:439 - Total trainable parameters: 14368
2026-10-02 06:53:56.680 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.494678, Accuracy: 9.38%
2026-10-02 06:54:12.035 | INFO     | t

In [19]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-02 07:22:46.280 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_arabic-characters_resnet18_seed43_epochs30_initrandom_betalr0.01.log
2026-10-02 07:22:46.334 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 07:23:02.460 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-02 07:23:02.460 | INFO     | __main__:main:394 - Requested initial stage betas: [0.889167, 0.712695, 0.705809, 0.943372]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-02 07:23:02.486 | INFO     | __main__:main:418 - Initial stage betas: [0.8891667127609253, 0.7126947641372681, 0.7058085799217224, 0.9433716535568237]
2026-10-02 07:23:02.487 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 07:23:02.487 | INFO     | __main__:main:439 - Total trainable parameters: 14368
2026-10-02 07:23:03.250 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.464369, Accuracy: 0.00%
2026-10-02 07:23:18.639 | INFO     |

In [ ]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-02 07:52:24.934 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_arabic-characters_resnet18_seed44_epochs30_initrandom_betalr0.01.log
2026-10-02 07:52:25.020 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 07:52:48.981 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-02 07:52:48.982 | INFO     | __main__:main:394 - Requested initial stage betas: [0.735544, 0.774853, 0.817674, 0.981063]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-02 07:52:49.018 | INFO     | __main__:main:418 - Initial stage betas: [0.7355440258979797, 0.7748528122901917, 0.8176735043525696, 0.9810633659362793]
2026-10-02 07:52:49.019 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 07:52:49.019 | INFO     | __main__:main:439 - Total trainable parameters: 14368
2026-10-02 07:52:50.054 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.488366, Accuracy: 3.12%
2026-10-02 07:53:05.835 | INFO     |